# 법령 PDF 자동 다운로드 노트북

- 샌드박스 법령 리스트를 읽어 법제처 사이트에서 PDF를 일괄 다운로드합니다.
- 실행 전 ChromeDriver 경로, 엑셀 위치 등을 **환경설정 셀**에서 수정하세요.

In [18]:
!pip install pandas openpyxl webdriver-manager

In [20]:
# 필수 라이브러리 로드
from __future__ import annotations
from pathlib import Path
from typing import List, Optional, Set
import logging
import time
import pandas as pd
from urllib.parse import quote_plus

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException, WebDriverException

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

In [28]:
# === 사용자 입력 영역 ===
BASE_DIR = Path.cwd()
LIST_FILE = BASE_DIR / "샌드박스_법령명_163개_리스트.xlsx"  # 엑셀 경로
COLUMN_NAME = "정식 법령명"  # 엑셀 내 법령명 컬럼명
DOWNLOAD_DIR = BASE_DIR / "law_doc_downloads"  # 다운로드 폴더

BASE_SEARCH_URL = "https://www.law.go.kr/lsSc.do?menuId=1&subMenuId=15&tabMenuId=81&query="
CHROMEDRIVER_PATH = Path("C:/Users/PCN_DT01/.wdm/drivers/chromedriver/win64/140.0.7339.207/chromedriver-win32/chromedriver.exe")  # 실제 chromedriver 경로로 교체
HEADLESS = False  # 서버 환경이면 True 권장
MAX_DOWNLOADS: Optional[int] = None  # 테스트용으로 n개만 받을 때 숫자 지정

SAVE_BUTTON_XPATH = "/html/body/form[2]/div[1]/div[2]/div[1]/div[3]/a[5]"
PDF_RADIO_XPATH = "/html/body/div[45]/div[2]/div/div/form/fieldset/div[3]/div[1]/div[3]"
DOC_RADIO_XPATH = "/html/body/div[45]/div[2]/div/div/form/fieldset/div[3]/div[1]/div[4]"
DOWNLOAD_BUTTON_XPATH = "/html/body/div[45]/div[2]/div/div/form/fieldset/div[3]/div[2]/a[1]"

DOWNLOAD_TIMEOUT_SEC = 180  # 파일 완료 대기시간
MODAL_TIMEOUT_SEC = 15  # 모달 표시 대기시간

DOWNLOAD_DIR.mkdir(exist_ok=True)

In [22]:
# === 헬퍼 함수 정의 ===
def sanitize_filename(raw_name: str) -> str:
    """파일 시스템에서 사용할 수 있도록 특수문자를 제거합니다."""
    invalid_chars = '<>:"/\\|?*'
    cleaned = ''.join(ch for ch in raw_name if ch not in invalid_chars).strip()
    return cleaned or "법령"


def load_law_names(list_path: Path, column_name: str) -> List[str]:
    """엑셀에서 법령명 목록을 읽어 문자열 리스트로 반환합니다."""
    if not list_path.exists():
        raise FileNotFoundError(f"엑셀 파일을 찾을 수 없습니다: {list_path}")
    df = pd.read_excel(list_path)
    if column_name not in df.columns:
        raise ValueError(f"엑셀에 '{column_name}' 컬럼이 없습니다. 실제 컬럼명을 확인하세요.")
    names = (
        df[column_name]
        .dropna()
        .astype(str)
        .str.strip()
    )
    result = [name for name in names if name]
    if not result:
        raise ValueError("다운로드할 법령명이 비어 있습니다.")
    return result


def build_driver(download_dir):
    options = Options()
    
    # 다운로드 경로 설정 등 기존 옵션 유지
    prefs = {
        "download.default_directory": str(download_dir),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)

    # 필요하면 headless
    # options.add_argument("--headless=new")

    # 🚨 Service / CHROMEDRIVER_PATH 제거
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(60)
    return driver


def wait_for_new_file(download_dir: Path, before_files: Set[Path], timeout_sec: int) -> Path:
    """다운로드 폴더에서 새롭게 생성된 파일이 나타날 때까지 대기합니다."""
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        current_files = {
            file_path for file_path in download_dir.glob('*')
            if file_path.is_file() and not file_path.name.endswith('.crdownload')
        }
        diff = current_files - before_files
        if diff:
            return max(diff, key=lambda p: p.stat().st_mtime)
        time.sleep(1)
    raise TimeoutException("다운로드 완료 파일을 찾지 못했습니다.")


def select_pdf_and_save(driver: webdriver.Chrome) -> None:
    """모달 내 PDF 라디오버튼과 저장 버튼을 순서대로 클릭합니다."""
    wait = WebDriverWait(driver, MODAL_TIMEOUT_SEC)
    pdf_radio = wait.until(EC.element_to_be_clickable((By.XPATH, PDF_RADIO_XPATH)))
    pdf_radio.click()
    save_btn = wait.until(EC.element_to_be_clickable((By.XPATH, DOWNLOAD_BUTTON_XPATH)))
    save_btn.click()


def download_single_law(
    driver: webdriver.Chrome,
    law_name: str,
    base_url: str,
    download_dir: Path,
) -> Path:
    """단일 법령에 대해 PDF 다운로드를 수행하고 파일 경로를 반환합니다."""
    encoded_name = quote_plus(law_name)
    target_url = f"{base_url}{encoded_name}"
    logging.info("검색 페이지 접속: %s", target_url)
    driver.get(target_url)
    wait = WebDriverWait(driver, 20)
    save_button = wait.until(EC.element_to_be_clickable((By.XPATH, SAVE_BUTTON_XPATH)))
    save_button.click()
    select_pdf_and_save(driver)
    before_files = {f.resolve() for f in download_dir.glob('*') if f.is_file()}
    downloaded_file = wait_for_new_file(download_dir, before_files, DOWNLOAD_TIMEOUT_SEC)
    target_name = sanitize_filename(law_name)
    final_path = download_dir / f"{target_name}.pdf"
    downloaded_file.rename(final_path)
    logging.info("다운로드 완료: %s -> %s", law_name, final_path)
    return final_path


def bulk_download(
    law_names: List[str],
    download_dir: Path,
    base_url: str,
    max_downloads: Optional[int] = None,
) -> dict:
    """법령명을 순회하며 다운로드하고 성공/실패 내역을 반환합니다."""
    target_list = law_names if max_downloads is None else law_names[:max_downloads]
    driver = build_driver(download_dir)
    successes, failures = [], []
    try:
        for idx, name in enumerate(target_list, start=1):
            logging.info("[%s/%s] 처리 중: %s", idx, len(target_list), name)
            try:
                path = download_single_law(driver, name, base_url, download_dir)
                successes.append(path)
            except Exception as exc:  # pylint: disable=broad-except
                logging.error("다운로드 실패 (%s): %s", name, exc)
                failures.append({"law_name": name, "error": str(exc)})
    finally:
        driver.quit()
    return {"successes": successes, "failures": failures}


In [29]:
# === 헬퍼 함수 정의 ===
def sanitize_filename(raw_name: str) -> str:
    """파일 시스템에서 사용할 수 있도록 특수문자를 제거합니다."""
    invalid_chars = '<>:"/\\|?*'
    cleaned = ''.join(ch for ch in raw_name if ch not in invalid_chars).strip()
    return cleaned or "법령"


def load_law_names(list_path: Path, column_name: str) -> List[str]:
    """엑셀에서 법령명 목록을 읽어 문자열 리스트로 반환합니다."""
    if not list_path.exists():
        raise FileNotFoundError(f"엑셀 파일을 찾을 수 없습니다: {list_path}")
    df = pd.read_excel(list_path)
    if column_name not in df.columns:
        raise ValueError(f"엑셀에 '{column_name}' 컬럼이 없습니다. 실제 컬럼명을 확인하세요.")
    names = (
        df[column_name]
        .dropna()
        .astype(str)
        .str.strip()
    )
    result = [name for name in names if name]
    if not result:
        raise ValueError("다운로드할 법령명이 비어 있습니다.")
    return result


def build_driver(download_dir):
    options = Options()
    
    # 다운로드 경로 설정 등 기존 옵션 유지
    prefs = {
        "download.default_directory": str(download_dir),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    options.add_experimental_option("prefs", prefs)

    # 필요하면 headless
    # options.add_argument("--headless=new")

    # 🚨 Service / CHROMEDRIVER_PATH 제거
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(60)
    return driver


def wait_for_new_file(download_dir: Path, before_files: Set[Path], timeout_sec: int) -> Path:
    """다운로드 폴더에서 새롭게 생성된 파일이 나타날 때까지 대기합니다."""
    deadline = time.time() + timeout_sec
    while time.time() < deadline:
        current_files = {
            file_path for file_path in download_dir.glob('*')
            if file_path.is_file() and not file_path.name.endswith('.crdownload')
        }
        diff = current_files - before_files
        if diff:
            return max(diff, key=lambda p: p.stat().st_mtime)
        time.sleep(1)
    raise TimeoutException("다운로드 완료 파일을 찾지 못했습니다.")


def select_doc_and_save(driver: webdriver.Chrome) -> None:
    """모달 내 Word(DOC/DOCX) 라디오버튼과 저장 버튼을 순서대로 클릭합니다."""
    wait = WebDriverWait(driver, MODAL_TIMEOUT_SEC)
    
    # Word용 라디오 버튼 클릭 (DOC_RADIO_XPATH 사용)
    doc_radio = wait.until(EC.element_to_be_clickable((By.XPATH, DOC_RADIO_XPATH)))
    doc_radio.click()
    
    # '저장' 버튼 클릭
    save_btn = wait.until(EC.element_to_be_clickable((By.XPATH, DOWNLOAD_BUTTON_XPATH)))
    save_btn.click()



def download_single_law(
    driver: webdriver.Chrome,
    law_name: str,
    base_url: str,
    download_dir: Path,
) -> Path:
    """단일 법령에 대해 Word(DOC/DOCX) 다운로드를 수행하고 파일 경로를 반환합니다."""
    encoded_name = quote_plus(law_name)
    target_url = f"{base_url}{encoded_name}"
    logging.info("검색 페이지 접속: %s", target_url)

    driver.get(target_url)
    wait = WebDriverWait(driver, 20)

    # 저장 버튼 클릭 (페이지 내 상단/하단 '저장' 버튼)
    save_button = wait.until(EC.element_to_be_clickable((By.XPATH, SAVE_BUTTON_XPATH)))
    save_button.click()

    # ⚠️ 다운로드 전, 기존 파일 목록 스냅샷
    before_files = {
        f.resolve()
        for f in download_dir.glob('*')
        if f.is_file() and not f.name.endswith('.crdownload')
    }

    # 모달에서 Word 라디오 버튼 선택 후 저장
    select_doc_and_save(driver)

    # 새로 생긴 파일이 나올 때까지 대기
    downloaded_file = wait_for_new_file(download_dir, before_files, DOWNLOAD_TIMEOUT_SEC)

    # 최종 파일명: 법령명 기반 + .docx (필요하면 .doc 로 변경)
    target_name = sanitize_filename(law_name)
    final_path = download_dir / f"{target_name}.docx"
    downloaded_file.rename(final_path)

    logging.info("다운로드 완료 (WORD): %s -> %s", law_name, final_path)
    return final_path



def bulk_download(
    law_names: List[str],
    download_dir: Path,
    base_url: str,
    max_downloads: Optional[int] = None,
) -> dict:
    """법령명을 순회하며 Word(DOC/DOCX)로 다운로드하고 성공/실패 내역을 반환합니다."""
    target_list = law_names if max_downloads is None else law_names[:max_downloads]
    driver = build_driver(download_dir)

    successes, failures = [], []

    try:
        for idx, name in enumerate(target_list, start=1):
            logging.info("[%s/%s] 처리 중: %s", idx, len(target_list), name)
            try:
                path = download_single_law(driver, name, base_url, download_dir)
                successes.append(path)
            except Exception as exc:  # pylint: disable=broad-except
                logging.error("다운로드 실패 (%s): %s", name, exc)
                failures.append({"law_name": name, "error": str(exc)})
    finally:
        driver.quit()

    return {"successes": successes, "failures": failures}

In [30]:
# === 실행 셀 ===
law_name_list = load_law_names(LIST_FILE, COLUMN_NAME)
summary = bulk_download(
    law_names=law_name_list,
    download_dir=DOWNLOAD_DIR,
    base_url=BASE_SEARCH_URL,
    max_downloads=MAX_DOWNLOADS,
)
summary

2025-11-25 13:41:39,499 INFO [1/163] 처리 중: 간선급행버스체계의 건설 및 운영에 관한 특별법
2025-11-25 13:41:39,504 INFO 검색 페이지 접속: https://www.law.go.kr/lsSc.do?menuId=1&subMenuId=15&tabMenuId=81&query=%EA%B0%84%EC%84%A0%EA%B8%89%ED%96%89%EB%B2%84%EC%8A%A4%EC%B2%B4%EA%B3%84%EC%9D%98+%EA%B1%B4%EC%84%A4+%EB%B0%8F+%EC%9A%B4%EC%98%81%EC%97%90+%EA%B4%80%ED%95%9C+%ED%8A%B9%EB%B3%84%EB%B2%95
2025-11-25 13:41:42,233 INFO 다운로드 완료 (WORD): 간선급행버스체계의 건설 및 운영에 관한 특별법 -> C:\work\03. 제안서\2025\AGI\데이터\law_doc_downloads\간선급행버스체계의 건설 및 운영에 관한 특별법.docx
2025-11-25 13:41:42,234 INFO [2/163] 처리 중: 감염병의 예방 및 관리에 관한 법률
2025-11-25 13:41:42,235 INFO 검색 페이지 접속: https://www.law.go.kr/lsSc.do?menuId=1&subMenuId=15&tabMenuId=81&query=%EA%B0%90%EC%97%BC%EB%B3%91%EC%9D%98+%EC%98%88%EB%B0%A9+%EB%B0%8F+%EA%B4%80%EB%A6%AC%EC%97%90+%EA%B4%80%ED%95%9C+%EB%B2%95%EB%A5%A0
2025-11-25 13:41:45,379 INFO 다운로드 완료 (WORD): 감염병의 예방 및 관리에 관한 법률 -> C:\work\03. 제안서\2025\AGI\데이터\law_doc_downloads\감염병의 예방 및 관리에 관한 법률.docx
2025-11-25 13:41:45,381 INFO [3/163]

{'successes': [WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/간선급행버스체계의 건설 및 운영에 관한 특별법.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/감염병의 예방 및 관리에 관한 법률.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/개인정보 보호법.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/개인정보 보호법 시행령.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/건강기능식품에 관한 법률.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/게임산업진흥에 관한 법률.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/고압가스 안전관리법.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/고압가스 안전관리법 시행규칙.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/고압가스 안전관리법 시행령.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/공중위생관리법.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloads/공중위생관리법 시행령.docx'),
  WindowsPath('C:/work/03. 제안서/2025/AGI/데이터/law_doc_downloa

In [24]:
# === 단위 테스트 셀 ===
import unittest
import tempfile

class HelperTests(unittest.TestCase):
    def test_sanitize_filename_removes_invalid_chars(self):
        self.assertEqual(sanitize_filename('테스트<>:"/\\|?* '), '테스트')

    def test_load_law_names_reads_column(self):
        sample_df = pd.DataFrame({COLUMN_NAME: ['법령A', ' ', None, '법령B']})
        with tempfile.NamedTemporaryFile(suffix=".xlsx", delete=False) as tmp:
            sample_df.to_excel(tmp.name, index=False)
            names = load_law_names(Path(tmp.name), COLUMN_NAME)
        self.assertEqual(names, ['법령A', '법령B'])

suite = unittest.TestLoader().loadTestsFromTestCase(HelperTests)
unittest.TextTestRunner(verbosity=2).run(suite)

test_load_law_names_reads_column (__main__.HelperTests.test_load_law_names_reads_column) ... ok
test_sanitize_filename_removes_invalid_chars (__main__.HelperTests.test_sanitize_filename_removes_invalid_chars) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.118s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

In [31]:
# === 누락 파일 재다운로드 셀 ===
# 누락된 PDF를 확인하고 필요한 항목만 재다운로드합니다.
from pprint import pprint

# 1) 최신 다운로드 현황 수집
law_name_list = load_law_names(LIST_FILE, COLUMN_NAME)
existing_files = {file_path.stem for file_path in DOWNLOAD_DIR.glob("*.pdf")}
missing_law_names = [name for name in law_name_list if sanitize_filename(name) not in existing_files]

print(f"총 {len(law_name_list)}건 중 {len(existing_files)}건 다운로드 완료")
print(f"현재 누락 건수: {len(missing_law_names)}")

retry_summary = None
if missing_law_names:
    print("누락된 항목 재다운로드 진행 중...")
    retry_summary = bulk_download(
        law_names=missing_law_names,
        download_dir=DOWNLOAD_DIR,
        base_url=BASE_SEARCH_URL,
        max_downloads=None,
    )
else:
    print("모든 파일이 이미 다운로드되었습니다.")

# 2) 재다운로드 이후 최종 상태 재확인
final_files = {file_path.stem for file_path in DOWNLOAD_DIR.glob("*.pdf")}
final_missing = [name for name in law_name_list if sanitize_filename(name) not in final_files]

print(f"재확인 후 누락 건수: {len(final_missing)}")
if final_missing:
    print("아직 남은 누락 항목:")
    pprint(final_missing)
else:
    print("모든 파일 다운로드 완료!")

retry_summary

총 163건 중 0건 다운로드 완료
현재 누락 건수: 163
누락된 항목 재다운로드 진행 중...


2025-11-25 14:16:36,417 INFO [1/163] 처리 중: 간선급행버스체계의 건설 및 운영에 관한 특별법
2025-11-25 14:16:36,418 INFO 검색 페이지 접속: https://www.law.go.kr/lsSc.do?menuId=1&subMenuId=15&tabMenuId=81&query=%EA%B0%84%EC%84%A0%EA%B8%89%ED%96%89%EB%B2%84%EC%8A%A4%EC%B2%B4%EA%B3%84%EC%9D%98+%EA%B1%B4%EC%84%A4+%EB%B0%8F+%EC%9A%B4%EC%98%81%EC%97%90+%EA%B4%80%ED%95%9C+%ED%8A%B9%EB%B3%84%EB%B2%95
2025-11-25 14:16:37,394 ERROR 다운로드 실패 (간선급행버스체계의 건설 및 운영에 관한 특별법): Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=142.0.7444.175)
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff6df80a235
	0x7ff6df562630
	0x7ff6df2f16dd
	0x7ff6df2c8e31
	0x7ff6df37a22e
	0x7ff6df39b3d2
	0x7ff6df33b0ed
	0x7ff6df33bf63
	0x7ff6df835d60
	0x7ff6df82fe8a
	0x7ff6df851005
	0x7ff6df57d71e
	0x7ff6df584e1f
	0x7ff6df56b7c4
	0x7ff6df56b97f
	0x7ff6df5518e8
	0x7ffc4260e8d7
	0x7ffc43d0c53c

2025-11-25 14:16:37,395 INFO [2/163] 처리 중: 감염병의 예방 및 관리에 관한 법률
2025-11-25 14:16:

KeyboardInterrupt: 

In [32]:
# === 누락 파일 재다운로드 셀 (WORD 버전: .doc / .docx) ===
from pprint import pprint

# 1) 최신 다운로드 현황 수집
law_name_list = load_law_names(LIST_FILE, COLUMN_NAME)

# 기존: *.pdf → 변경: *.doc / *.docx 모두 인식
existing_files = {
    file_path.stem
    for file_path in DOWNLOAD_DIR.glob("*.doc*")  # .doc, .docx 모두 포함
}

missing_law_names = [
    name for name in law_name_list
    if sanitize_filename(name) not in existing_files
]

print(f"총 {len(law_name_list)}건 중 {len(existing_files)}건 다운로드 완료")
print(f"현재 누락 건수: {len(missing_law_names)}")

retry_summary = None
if missing_law_names:
    print("누락된 항목 재다운로드 진행 중 (WORD 파일 기준)...")
    retry_summary = bulk_download(
        law_names=missing_law_names,
        download_dir=DOWNLOAD_DIR,
        base_url=BASE_SEARCH_URL,
        max_downloads=None,
    )
else:
    print("모든 파일이 이미 다운로드되었습니다. (WORD 파일 기준)")

# 2) 재다운로드 이후 최종 상태 재확인
final_files = {
    file_path.stem
    for file_path in DOWNLOAD_DIR.glob("*.doc*")  # .doc, .docx 기준
}
final_missing = [
    name for name in law_name_list
    if sanitize_filename(name) not in final_files
]

print(f"재확인 후 누락 건수: {len(final_missing)}")
if final_missing:
    print("아직 남은 누락 항목:")
    pprint(final_missing)
else:
    print("모든 파일 다운로드 완료! (WORD 파일 기준)")

retry_summary


총 163건 중 148건 다운로드 완료
현재 누락 건수: 15
누락된 항목 재다운로드 진행 중 (WORD 파일 기준)...


2025-11-25 14:17:49,511 INFO [1/15] 처리 중: 건축법
2025-11-25 14:17:49,511 INFO 검색 페이지 접속: https://www.law.go.kr/lsSc.do?menuId=1&subMenuId=15&tabMenuId=81&query=%EA%B1%B4%EC%B6%95%EB%B2%95
2025-11-25 14:17:52,131 ERROR 다운로드 실패 (건축법): Message: element click intercepted: Element <a href="javascript:;" class="btn_c type3 typeB" id="bdySaveBtn" onclick="bdySavePrint(0, this ,0, '');return false;" title="저장">...</a> is not clickable at point (994, 274). Other element would receive the click: <div class="l_bx">...</div>
  (Session info: chrome=142.0.7444.175); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff6df80a235
	0x7ff6df562630
	0x7ff6df2f16dd
	0x7ff6df3524c9
	0x7ff6df34fe4e
	0x7ff6df34cd71
	0x7ff6df34bc10
	0x7ff6df33d3a8
	0x7ff6df372b0a
	0x7ff6df33cc36
	0x7ff6df39baba
	0x7ff6df33b0ed
	0x7ff6df33bf63
	0x7ff6df835d60
	0

재확인 후 누락 건수: 15
아직 남은 누락 항목:
['건축법',
 '건축법 시행령',
 '국토의 계획 및 이용에 관한 법률 시행령',
 '도로교통법',
 '먹는물관리법',
 '산업집적활성화 및 공장설립에 관한 법률',
 '상법',
 '식품위생법',
 '식품위생법 시행령',
 '의료법 시행규칙',
 '자동차관리법',
 '자동차관리법 시행규칙',
 '자본시장과 금융투자업에 관한 법률 시행령',
 '자전거 이용 활성화에 관한 법률',
 '장애인복지법']


{'successes': [],
 'failures': [{'law_name': '건축법',
   'error': 'Message: element click intercepted: Element <a href="javascript:;" class="btn_c type3 typeB" id="bdySaveBtn" onclick="bdySavePrint(0, this ,0, \'\');return false;" title="저장">...</a> is not clickable at point (994, 274). Other element would receive the click: <div class="l_bx">...</div>\n  (Session info: chrome=142.0.7444.175); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception\nStacktrace:\nSymbols not available. Dumping unresolved backtrace:\n\t0x7ff6df80a235\n\t0x7ff6df562630\n\t0x7ff6df2f16dd\n\t0x7ff6df3524c9\n\t0x7ff6df34fe4e\n\t0x7ff6df34cd71\n\t0x7ff6df34bc10\n\t0x7ff6df33d3a8\n\t0x7ff6df372b0a\n\t0x7ff6df33cc36\n\t0x7ff6df39baba\n\t0x7ff6df33b0ed\n\t0x7ff6df33bf63\n\t0x7ff6df835d60\n\t0x7ff6df82fe8a\n\t0x7ff6df851005\n\t0x7ff6df57d71e\n\t0x7ff6df584e1f\n\t0x7ff6df56b7c4\n\t0x7ff6df56b97f\n\t0x7ff6df5518e8\n\t0x7ff